# Silver Exploration Notebook


## Imports

In [ ]:
import os
import findspark

from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col,
    lit,
    when,
    to_timestamp,
    from_utc_timestamp,
    dayofweek,
    hour,
    month,
    input_file_name,
    regexp_extract,
    arrays_zip,
    explode,
    size,
    count,
    length,
    min,
    max,
    avg,
    slice,
    regexp_extract,
    desc,
    unix_micros,
    abs,
    split,
    element_at,
    countDistinct,
    sum as spark_sum
)

## Start Spark

In [ ]:
findspark.init()

spark = (
    SparkSession.builder
    .appName("silver-exploration")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

In [ ]:
RAW = os.path.expanduser("~/spark-lab-data/dev/project/raw")
print(RAW)

## Read Bike-sharing dataset
+ Adding city and month

In [ ]:
# Bike-sharing data
trips_raw = (
    spark.read
    .option("header", True)
    .csv(f"{RAW}/trips/*/*.csv")
    .withColumn("src", input_file_name())
)

trips_raw = (
    trips_raw
    .withColumn("city", element_at(split(col("src"), "/"), -2))
    .withColumn("year_month", element_at(split(col("src"), "/"), -1))
    .withColumn("year_month",split(col("year_month"), r"\.").getItem(0))
)

trips_raw.select("city", "year_month").show(5)

## Exporation of Bike-sharing dataset

In [ ]:
trips_raw.printSchema()
trips_raw.show(5, truncate=False)
print("Total rows:", trips_raw.count())
print("Columns:", trips_raw.columns)

- We have more than 8 460 000 biketrips in Oslo, Trondheim and Bergen.
- The columns in the Bike-sharing dataset include: `started_at`, `endend_at`, `duration`, `start_station_id`, `start_station_name`, `start_station_description`, `start_station_latitude`, `start_station_longitude`, `end_station_id`, `end_station_name`, `end_station_description`, `end_station_latitude`, `end_station_longitude`. We also have the added `src`, `city` and `year_month`.

#### Data Types
The 16 columns are stored as string data types. This is because PySpark reads CSV columns as strings by default. However, Several columns represent numeric values or timestamps. We will therefore convert these columns to appropriate data types. 

- Timestamps: `started_at`and `ended_at` -> timestamp
- Trip duration: `duration` -> integer(maybe double?)
- Coorinates: `latitude`and `longitude`-> double

Maybe:
- StationIDs: `start_station_id` and `stop_station_id` -> keep as string or convert to integer

From the table above we can see the ID is stored as a numeric value, but we would argue to keep them as a string as they are identifiers rather than numeric measurements. This also provides a greater flexibility if future datasets contains letters(e.g. ST123), without requiring changes to the data type. 


### Investigating missing values

In [ ]:
trips_raw.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in trips_raw.columns
]).show()

#### Missing values
The `start_station_description` column contains 11 000 missing values, while `end_station_description` contains around 7000 missing values.

#### Is this detrimental to our project or analysis?
These columns are not truly important for the type of analysing we want to do. Our main objective is to investigate how weather conditions influence bike-sharing usage. Since station ID and names are available, the missing descriptions do not prevent us from identifying stations and analyzing trips. 

Although they are not required for our current objectives, they could be valuable for other purposes later. In the Gold layer, we can simply exclude columns that are not relevant to are final analytical dataset. 

### Investigating duplicates

In [ ]:
print("Total rows:", trips_raw.count())
print("Duplicate rows:", trips_raw.count() - trips_raw.distinct().count())

Out of the 8 428 152 trips, 25 additional duplicates. 

#### Are we sure these are duplicates?

We can not be certain that these are dublicates because the trips do not contain any identifier. These records do however contain identical entries across all 16 collumns, including timestamp with microseconds precision. It is highly unlikely that separate trips would share all these values.

We consider therefore these records to be duplicate entries, potentially from the source data or overlapping records during the ingestion process. The exact cause has not been investigated further, but we want to remove the additional copies. 

Given that our dataset contain over 8 million trips, removing 25 duplicate records will have a negligible impact on the overall analysis. If the dataset was significanlty smaller, further investigation would be important as removing even a small number of records could have a greater influence on the result.

In [ ]:
duplicate_cols = [
    c for c in trips_raw.columns
    if c not in ["src", "year_month"]
]

trips_silver = trips_raw.dropDuplicates(duplicate_cols)

print("Total rows:", trips_raw.count())
print("Duplicate rows:", trips_raw.count() - trips_silver.count())

### Investigatin values and distributions

#### First Duration
Lets start with convert the `duration` column to a double data type. 

In [ ]:
trips_silver = (
    trips_silver
    .withColumn("duration", col("duration").cast("double"))
)

# Did the convertion work?
trips_silver.filter(
    col("duration").isNull() 
).count()


In [ ]:

trips_silver.select("duration").summary("min", "25%", "50%", "75%", "max").show()

trips_silver.select(spark_sum(when(col("duration") <= 0, 1).otherwise(0)).alias("non_positive"),
         spark_sum(when(col("duration") < 60, 1).otherwise(0)).alias("under_60s"),
         spark_sum(when(col("duration").isNull(), 1).otherwise(0)).alias("null_duration")).show()

#### Durations to investigate ferther
- 2 trips has a negative or a duration of 0.
- Is it possible that a trip last around 22 hours? (79790/3600 = 22)

In [ ]:
# Trips with durations smaller than 1 second
trips_silver.filter(
    col("duration").cast("double") <= 1
).show(truncate=False)

print("How many trips have a duration of 1 second or less?", trips_silver.filter(col("duration").cast("double") <= 1).count())

### Extremely Short Trips
29 trips with a recorded duration of one second or less, including two trips with a duration of zero seconds. 

With the table above these trips have a different start and end station which makes it physically impossible to complete the journeys within that timeframe.

These records likely represent data quality issues that need futher investigation.

Possible solution: 
- Find the mean trip duration from one statiotn to the other and compare it to minimum duration record.
- Flag trips under a spesific timeframe. 


In [ ]:
station_durations = (
    trips_silver
    .groupBy("city", "start_station_id", "end_station_id")
    .agg(
        count("*").alias("num_trips"),
        avg("duration").alias("mean_duration"),
        min("duration").alias("min_duration")
    )
)

station_durations.filter(
    col("num_trips") >= 10
).orderBy("min_duration").show(20, truncate=False)

### What does the table above tell us?
Although the mean duration does not accually tell us the minimum possible duration between two stations, it provides a useful indication of what a typical trip looks like. 

The table shows station pairs with hundres and thousands of bike-trips, where the average duration is several minutes, yet the shortest recorded time is only one second. This strongly suggest that some values are incorrect or unrealistic. 

On another note this does make me question the dataset and its accuracy. Are these simply a small number of isolated data quality issues, or could they indicate more problems with the data?

For now let's atleast flag trips with a duration smaller than one second. 

In [ ]:
# Flag trips that are under 1 second
trips_silver = trips_silver.withColumn(
    "short_trip_flag",
    col("duration") <= 1
)

### Trips with a long duration

In [ ]:
print("Number of trips longer than 12 hours: ", trips_silver.filter(col("duration") > 12 * 3600).count())

trips_silver.select(
    "started_at",
    "ended_at",
    "duration"
).orderBy(desc("duration")).show(15, truncate=False)

26/11/15 12:06:48 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:06:48 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:06:48 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:06:48 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:06:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:06:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:06:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:06:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/15 12:06:50 WARN RowBasedKeyValueBatch: Calling spill() on

Could a city bike realistically be used for appocimately 22 hours?

Accoring to Oslo Bysykkel, a fully charged electric bike has an estimate range of around 70 kilometers. The battery does not necessarily limit the recorded trip duration, as the bike can remain parked while the rental is still active and you can manually ride the bike if the battery runs out. 

If the dataset included distance travelled we could better investigate whether the bicycle was actively used.

Possible explainations for the long duration
- The user forgot to return the bike or didn't properly complete the return process.
- The bike was parked for several hours while the rental was still active. 
- The recorded timestamp contains an error. 
- Maybe others?

In [ ]:
print("Over 3 hours:", trips_silver.filter(col("duration") > 3 * 60 * 60).count())
print("Over 6 hours:", trips_silver.filter(col("duration") > 6 * 60 * 60).count())
print("Over 12 hours:", trips_silver.filter(col("duration") > 12 * 60 * 60).count())

(3184/8460580) * 100 = 0.0376 %

Only 0.038% of all bike trips last longer than 3 hours, inducating such trips are extremely rare.

### Investigating the timestamp format

In [ ]:
trips_silver.groupBy(length("started_at").alias("len")).count().orderBy("len").show()
trips_silver.select("started_at", "ended_at", "duration").distinct().limit(10).show(truncate=False)

#### Converting the string to UTC timestamp
The `started_at` and `ended_at` timestamps are recorded in UTC, as indicated by `+00:00`.

For example:
- `started_at`: 2022-08-01 05:14:13.781000+00:00
- `ended_at`: 2022-08-01 05:26:27.585000+00:00

In [ ]:
# Setting the spark session timezone to UTC
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Converting the started_at and ended_at columns to timestamp type
trips_silver = (
    trips_silver
    .withColumn("started_ts", to_timestamp(col("started_at")))
    .withColumn("ended_ts", to_timestamp(col("ended_at")))
)

# Let's compare the converted timestamps
trips_silver.select(
    "started_at",
    "started_ts",
    "ended_at",
    "ended_ts"
).show(10, truncate=False)

In [ ]:
# Did the convertion work?
trips_silver.filter(
    col("started_ts").isNull() |
    col("ended_ts").isNull()
).count()

### Converting Timestamps to Norwegian Local Time

We convert these timestamps to Norwegian local time (`Europe/Oslo`), accounting for daylight saving time. The original UTC timestamps are preserved for further processing.

In [ ]:
trips_silver = (
    trips_silver
    .withColumn(
        "started_local",
        from_utc_timestamp(
            to_timestamp(col("started_ts")),
            "Europe/Oslo"
        )
    )
    .withColumn(
        "ended_local",
        from_utc_timestamp(
            to_timestamp(col("ended_ts")),
            "Europe/Oslo"
        )
    )
)

trips_silver.select("started_ts","started_local","ended_ts","ended_local").show(10, truncate=False)

#### Does duration correspond to the start and end time?

In [ ]:
trips_silver.select(
    "started_local",
    "ended_local",
    "duration",
    # does started_at and ended_at match the duration?
    (
        (
            unix_micros("ended_ts") -
            unix_micros("started_ts")
        ) / 1000000
    ).alias("duration_check")

).show(5, truncate=False)


From the table above it looks like the duration always round down.

for example:
- 363 and 363.712
- 703 and 703
- 778 and 778.756

Let's investigate the duration a bit further.

In [ ]:
trips_check = trips_silver.withColumn(
    "duration_diff",
    abs(
        (unix_micros("ended_ts")
         - unix_micros("started_ts")) / 1_000_000
        - col("duration").cast("double")
    )
)

trips_check.select("duration_diff").summary().show()

Since the maximum difference are below one second, we consider the recorded durations consistent with the timestamps. 

### Investigating latitude and longitude
first converting the data types to double

In [ ]:
trips_silver = (
    trips_silver
    .withColumn("start_station_latitude", col("start_station_latitude").cast("double"))
    .withColumn("start_station_longitude", col("start_station_longitude").cast("double"))
    .withColumn("end_station_latitude", col("end_station_latitude").cast("double"))
    .withColumn("end_station_longitude", col("end_station_longitude").cast("double"))
)

# Did the convertion work?
trips_silver.filter(
    col("start_station_latitude").isNull() |
    col("start_station_longitude").isNull() |
    col("end_station_latitude").isNull() |
    col("end_station_longitude").isNull()
).count()

#### Valid cordination for latitude and longitude in Norway:
- Norwegian latitude: 57 - 71
- Norwegian longitude: 4 - 31

In [ ]:
trips_silver.filter(
    (col("start_station_latitude") < 57) |
    (col("start_station_latitude") > 71) |
    (col("end_station_latitude") < 57) |
    (col("end_station_latitude") > 71) |
    (col("start_station_longitude") < 4) |
    (col("start_station_longitude") > 31) |
    (col("end_station_longitude") < 4) |
    (col("end_station_longitude") > 31)
).count()

In [ ]:
trips_silver.select(
    min("start_station_latitude").alias("min_lat"),
    max("start_station_latitude").alias("max_lat"),
    min("start_station_longitude").alias("min_lon"),
    max("start_station_longitude").alias("max_lon")
).show()

The latitude and logitude values appear to reasonable based on the geographical locations of the three cities. Oslo is the city most southern, while Trondheim is the city most nothern. Bergen is the city most western, while oslo is the city most eastern. 

|   City       |   latitude    |   longitude   |
|--------------|---------------|---------------|
|  Oslo:       |     59.91     |     10.75     |  
|  Bergen:     |     60.39     |      5.32     |
|  Trondheim:  |     63.43     |     10.40     |

### Number of trips divided by the different cities

In [ ]:
trips_silver.groupBy("city").count().show()

Oslo accounts for approximately 68% of all trips in the dataset, so the analysis of the trips may be dominated by the travel patterns in Oslo.

#### Number of trips devided by city and stations

In [ ]:
trips_silver.groupBy("city", "start_station_id", "end_station_id").count().show(200)

From the table above it looks like the stationIDs might use the same identifications accross the cities. So the ID alone won't be possible to identify which city the trip was in.  

Let's investigate this a bit ferther:

In [ ]:
trips_silver.groupBy("start_station_id").agg(
    countDistinct("city").alias("num_cities")
).filter(
    col("num_cities") > 1
).show()

The investigantion shows that no start station id is shared across cities in this current dataset. I do think including cities when grouping by station IDs to ensure robustness against potential ovelaps in future data and it gives a better overview. 

#### Number of trips divided by city and month

In [ ]:
trips_silver.groupBy("city", "year_month").count().orderBy(
    "city", "year_month"
).show(200)

### Summary and proposed Silver tranformations
Based on this explorations analysis, this is the following transformations we propose for the Silver layer:
- Extract city and year-month information from the source file path.
- Convert trip duration and geographical coordinates to double. 
- Converting start and and timestamps to UTC timestamps.
- Derive Norwegian local timestamps for temporal analysis. 
- Remove duplicate trips records. 
- Perserve station IDs as strings. 
- Perserve missing station descriptions, as they do not affect the main analytical objective.
- Flag trips with a recorded duration of one second. (Maybe even more?)
- Validate recorded durations against the start and end timestamps. 
- Validate geographical coordinates.